In [0]:
%pip install mlxtend

## 1. MBA using non-distributed/non-parallelized Python MLXtend
Because we can't run PySpark ML functions in a shared cluster, we're testing with MLXtend, an open-source python library for performing a market basket analysis. At most, on an 8-core driver node, we can only run about 1M rows of data. 

In [0]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

support_threshold = 0.005
confidence_threshold = 0.03

# grab a sample of data to validate
sample_df = sdf_cleaned.sample(withReplacement=False, fraction=1000000/sdf_prepared.count()).toPandas()

# One-hot encode our order_products 
dataset = sample_df['order_product_list'].to_list()
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df_ohe = pd.DataFrame(te_ary, columns=te.columns_)

# Apply the Apriori algorithm.
# Adjust min_support according to your data and desired threshold. 
# Support is the minimum proportion of transaction that an item must show up in to be considered as a part of the rule set
frequent_itemsets = apriori(df_ohe, min_support=support_threshold, use_colnames=True)
print("\nFrequent Itemsets:")
print(frequent_itemsets)

# Generate association rules from the frequent itemsets.
# Here, we set the metric to 'confidence' with a minimal confidence threshold.
rules = association_rules(frequent_itemsets, min_threshold=confidence_threshold, metric="confidence")
print("\nAssociation Rules:")
print(rules)

In [0]:
# This is what will be stashed into our PyFunc for model serving
# Function to recommend additional items given a transaction and the generated rules
def recommend_additional_items(transaction: set, rules_df):
    """
    Given a set of items (transaction) and a DataFrame of association rules,
    return a set of recommended items based on rules whose antecedents are
    a subset of the transaction.
    """
    recommendations = set()
    # Loop through each rule in the rules DataFrame
    for _, rule in rules_df.iterrows():
        antecedents = rule['antecedents']  # This is a frozenset
        consequents = rule['consequents']  # This is a frozenset
        # Check if the transaction contains all items in the antecedents
        if antecedents.issubset(transaction):
            # Recommend only items not already in the transaction
            recommendations = recommendations.union(consequents - transaction)
    return recommendations


In [0]:
current_transaction = {'Secret Recipe Fries', 'Cole Slaw', 'Chicken Sandwich'}

# Get recommendations based on current transaction
recommended_items = recommend_additional_items(current_transaction, rules)

print(f"Based on the current transaction {current_transaction} you might also consider adding: {recommended_items}")